# 🛠️ Pipeline de Dados - Parte 2: ETL e Data Warehouse

**Objetivo:** Extrair os dados sintéticos gerados em CSV, tratá-los e carregá-los em um banco de dados relacional MySQL (Carga do Data Warehouse).  
**Tecnologias:** Python, Pandas, MySQL Connector.

> **Contexto de Engenharia:** Na Parte 1, geramos os dados brutos. Agora, precisamos construir o pipeline de ingestão. O desafio aqui é garantir a **Integridade Referencial** (carregar as dimensões antes das tabelas fato) e garantir **Performance**. Bancos de dados podem derrubar a conexão se tentarmos inserir milhões de linhas de uma vez. Para evitar isso, implementamos uma arquitetura de carga em lotes (*Chunking*).

In [1]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import mysql.connector
from mysql.connector import Error
import math

In [3]:
def create_server_connection(host_name, user_name, user_password, db_name):
    connection = None
    try:
        connection = mysql.connector.connect(
            host = host_name,
            user = user_name,
            passwd=user_password,
            database=db_name
        )
        print(f'sucesso, conectado ao banco de dados {db_name}')
    except Error as err:
        print(f"erro de conexão: {err}")
    
    return connection

host = "localhost"
usuario = "root"
senha = "12345678"
banco_de_dados = "saas_dw"

#testando a conexão
conn = create_server_connection(host, usuario, senha, banco_de_dados)
print(conn)

sucesso, conectado ao banco de dados saas_dw


## 1. Funções Core do Pipeline (Limpeza e Ingestão)
Criamos ferramentas modulares para o pipeline. 
* A primeira função resolve um conflito de tipagem: O Pandas lê valores vazios como `NaN`, mas o banco SQL exige o formato `NULL`.
* A segunda função é o motor de inserção. Para garantir consistência e evitar o erro *Lost connection to MySQL*, a ingestão não é feita de uma vez, mas dividida em lotes menores seguros (Chunks).

In [ ]:
def clean_data_for_mysql(df):
    """
    O Pandas lê espaços vazios nos arquivos CSV como 'NaN' (Not a Number).
    Mas o MySQL não entende 'NaN', ele entende 'NULL' (que no Python é o 'None').
    Essa função varre a tabela e troca tudo para o padrão correto do banco.
    """
    df_cleaned = df.where(pd.notnull(df), None)

    dados_em_tuplas = [tuple(x) for x in df_cleaned.to_numpy()]

    return dados_em_tuplas


def execute_many_queries(connection, query, data):
    """ 
    Recebe a conexão, o comando SQL de INSERT e a lista de dados,
    e dispara para o MySQL utilizando processamento em lotes (chunking).      
    """
    cursor = connection.cursor()
    
    chunk_size = 5000
    total_inserted = 0
    
    try:
        #divide a lista em pedaços
        for i in range(0, len(data), chunk_size):
            chunk = data[i:i + chunk_size]
            cursor.executemany(query, chunk)
            connection.commit()
            total_inserted += cursor.rowcount
            
        print(f"sucesso! {total_inserted} linhas foram inseridas na tabela em lotes de {chunk_size}.")

    except Error as err:
        print(f"Erro durante a inserção: '{err}'")
    finally:
        cursor.close()


ferramenta de ETL criada e pronta para uso!


## 2. Fase de Carga: Tabelas de Dimensão
Em um modelo *Star Schema*, as tabelas de dimensão formam as pontas da estrela. Elas devem ser inseridas primeiro no banco de dados porque contêm as Chaves Primárias (IDs) que serão referenciadas mais tarde pelas tabelas Fato.

In [ ]:
print("iniciando o carregamento das Tabelas de Dimensão\n")

#carregando dim_plans
df_plans = pd.read_csv('dim_plans.csv')

# limpando os dados com a nossa ferramenta
plans_data = clean_data_for_mysql(df_plans)

# LOAD
query_plans = """
    INSERT INTO dim_plans (plan_id, plan_name, monthly_price)
    VALUES (%s, %s, %s)
"""
print("Injetando dim_plans...")
execute_many_queries(conn, query_plans, plans_data)

iniciando a ingestão das Tabelas de Dimensão

Injetando dim_plans...
sucesso! 3 linhas foram inseridas na tabela.


In [ ]:
#carregando dim_calendar
df_calendar = pd.read_csv('dim_calendar.csv')
calendar_data = clean_data_for_mysql(df_calendar)

query_calendar = """
    INSERT INTO dim_calendar (calendar_date, year, month, quarter, day_of_week)
    VALUES (%s, %s, %s, %s, %s)
"""
print("Injetando dim_calendar...")
execute_many_queries(conn, query_calendar, calendar_data)

Injetando dim_calendar...
sucesso! 1461 linhas foram inseridas na tabela.


In [ ]:
# carregando dim_customers

df_customers = pd.read_csv('dim_customers.csv')
customers_data = clean_data_for_mysql(df_customers)

query_customers = """
    INSERT INTO dim_customers (customer_id, tax_id, company_name, industry, state, company_size)
    VALUES (%s, %s, %s, %s, %s, %s)
"""
print("Injetando dim_customers...")
execute_many_queries(conn, query_customers, customers_data)

print("\n Dimensões carregadas com sucesso!")

Injetando dim_customers...
sucesso! 500 linhas foram inseridas na tabela.

 Dimensões carregadas com sucesso!


## 3. Fase de Carga: Tabelas Fato (Transacionais)
Com as entidades de negócio criadas, iniciamos a ingestão do volume pesado: o histórico de contratos e as movimentações de fluxo de caixa. Como essas tabelas crescem exponencialmente com o tempo, a técnica de *chunking* que configuramos na nossa ferramenta de ingestão sera ultilizada aqui.

In [ ]:
print("Iniciando o carregamento das tabelas fato\n")

# carregando fact_subscriptions
df_subscriptions = pd.read_csv('fact_subscriptions.csv')
subscriptions_data = clean_data_for_mysql(df_subscriptions)

query_subscriptions = """
    INSERT INTO fact_subscriptions (
        subscription_id, customer_id, plan_id, 
        start_date, cancel_date, subscription_status
    )
    VALUES (%s, %s, %s, %s, %s, %s)
"""
print("Injetando fact_subscriptions (Contratos)...")
execute_many_queries(conn, query_subscriptions, subscriptions_data)


Iniciando o carregamento das tabelas fato

Injetando fact_subscriptions (Contratos)...
sucesso! 500 linhas foram inseridas na tabela.


In [9]:
# injetando fact_payments
df_payments = pd.read_csv('fact_payments.csv')
payments_data = clean_data_for_mysql(df_payments)

query_payments = """
    INSERT INTO fact_payments (
        payment_id, subscription_id, due_date, 
        payment_date, payment_status, payment_method, amount_paid
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""
print("Injetando fact_payments (Fluxo de Caixa)...")
execute_many_queries(conn, query_payments, payments_data)

print("\nprocesso de ETL concluido! Seu Data Warehouse está totalmente pronto.")

Injetando fact_payments (Fluxo de Caixa)...
sucesso! 8398 linhas foram inseridas na tabela.

processo de ETL concluido! Seu Data Warehouse está totalmente pronto.
